# Session-Based Observability

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/session-observability)

Tag every LLM span with `using_user()` and `using_session()` so a multi-turn conversation groups into one filterable session in the Future AGI Tracing dashboard, instead of appearing as unrelated spans.

## Install

Start with a fresh Python 3.12 runtime. Set the named keys in your environment or Colab Secrets, or enter them at the hidden prompts.

In [ ]:
%pip install fi-instrumentation-otel==1.5.1 traceAI-openai==0.1.16 "openai>=1.69,<3"

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, RuntimeError):
        value = None
    except Exception as exc:
        if type(exc).__name__ not in {"SecretNotFoundError", "NotebookAccessError"}:
            raise
        value = None
    value = value or getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value

for name in ['FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY']:
    load_secret(name)

os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")

## Register the tracer and instrument OpenAI

In [ ]:
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from traceai_openai import OpenAIInstrumentor
from openai import OpenAI

trace_provider = register(
    project_type=ProjectType.OBSERVE,
    project_name="chatbot-session-demo",
    verbose=False,
)

OpenAIInstrumentor().instrument(tracer_provider=trace_provider)

client = OpenAI()

## Tag a single request with user and session context

In [ ]:
# No user/session context: this span is traced but ungrouped
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What should I pack for a rainy day hike?"}],
)
print(response.choices[0].message.content)

In [ ]:
from uuid import uuid4
from fi_instrumentation import using_user, using_session

run_id = uuid4().hex
user_id = f"hiker-{run_id}"
session_id = f"packing-{run_id}"

with using_user(user_id), using_session(session_id):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "What should I pack for a rainy day hike?"}],
    )
    print(response.choices[0].message.content)

## Run a multi-turn conversation

In [ ]:
from fi_instrumentation import using_user, using_session

def run_conversation(user_id: str, session_id: str) -> None:
    """Run a 3-turn conversation. Every span shares the same user and session IDs."""

    turns = [
        "Which layers would you recommend for a wet day hike?",
        "What if I get too warm on the climb?",
        "Summarize the packing list in one sentence.",
    ]

    conversation_history = []

    with using_user(user_id), using_session(session_id):
        for turn_number, user_message in enumerate(turns, start=1):
            conversation_history.append({"role": "user", "content": user_message})

            # Each call is auto-traced with the same user.id and session.id
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=conversation_history,
            )

            assistant_message = response.choices[0].message.content
            conversation_history.append({"role": "assistant", "content": assistant_message})
            print(f"Turn {turn_number}: {assistant_message[:80]}...")


run_conversation(user_id=user_id, session_id=f"layers-{run_id}")

## Add per-turn metadata

In [ ]:
from fi_instrumentation import using_user, using_session, using_metadata

def run_conversation_with_metadata(user_id: str, session_id: str) -> None:
    """Same conversation loop, with per-turn metadata attached to each span."""

    turns = [
        {"message": "Which layers would you recommend for a wet day hike?", "stage": "opening"},
        {"message": "What if I get too warm on the climb?", "stage": "deepening"},
        {"message": "Summarize the packing list in one sentence.", "stage": "closing"},
    ]

    conversation_history = []

    with using_user(user_id), using_session(session_id):
        for turn_number, turn in enumerate(turns, start=1):
            conversation_history.append({"role": "user", "content": turn["message"]})

            # Per-turn metadata is scoped to this span only
            turn_metadata = {
                "turn_number": turn_number,
                "conversation_stage": turn["stage"],
                "total_turns": len(turns),
            }

            with using_metadata(turn_metadata):
                response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=conversation_history,
                )

            assistant_message = response.choices[0].message.content
            conversation_history.append({"role": "assistant", "content": assistant_message})
            print(f"Turn {turn_number} [{turn['stage']}]: {assistant_message[:80]}...")


run_conversation_with_metadata(user_id=user_id, session_id=f"stages-{run_id}")

## Flush and verify the grouped session

In [ ]:
if not trace_provider.force_flush():
    raise TimeoutError("Timed out while flushing spans")
print(f"Look for stages-{run_id} in chatbot-session-demo")

Open **Sessions** in `chatbot-session-demo`. Look for the printed `stages-` session name and three traces, one for each stage.